In [ ]:
# ==============================================================================
# SCRIPT 1 (PYTHON): Parallel Regression & Japanese-4D Integration
# ==============================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from joblib import Parallel, delayed
import time
import os


def load_all_abundance_df():
    """Load AllCombinedSpProfile from CSV, or fall back to RData when CSV is empty."""
    csv_path = "AllCombinedSpProfile.csv"

    if os.path.exists(csv_path):
        try:
            csv_df = pd.read_csv(csv_path, index_col=0)
            if not csv_df.empty and csv_df.shape[1] > 0:
                print(f"Loaded abundance matrix from {csv_path}")
                return csv_df.fillna(0)
            print(f"{csv_path} exists but is empty; falling back to RData...")
        except pd.errors.EmptyDataError:
            print(f"{csv_path} is empty; falling back to RData...")
    else:
        print(f"{csv_path} not found; falling back to RData...")

    try:
        import pyreadr
    except ImportError as exc:
        raise ImportError(
            "pyreadr is required to read AllCombinedSpProfile from RData. "
            "Install with: pip install pyreadr"
        ) from exc

    rdata_candidates = [
        "Required_Data_FEBS_Pipeline.RData",
        "Required_Data_FEBS_Pipepline.RData"
    ]

    for rdata_path in rdata_candidates:
        if not os.path.exists(rdata_path):
            continue

        print(f"Trying to load abundance matrix from {rdata_path}...")
        r_objects = pyreadr.read_r(rdata_path)

        if "AllCombinedSpProfile" not in r_objects:
            continue

        all_abundance_df = r_objects["AllCombinedSpProfile"]
        if not isinstance(all_abundance_df, pd.DataFrame) or all_abundance_df.empty:
            raise ValueError(
                f"AllCombinedSpProfile in {rdata_path} is missing or empty."
            )

        print(f"Loaded abundance matrix from {rdata_path} (object: AllCombinedSpProfile)")
        return all_abundance_df.fillna(0)

    raise FileNotFoundError(
        "Could not load AllCombinedSpProfile. Checked AllCombinedSpProfile.csv and "
        "RData files: Required_Data_FEBS_Pipeline.RData / Required_Data_FEBS_Pipepline.RData"
    )


def load_target_42_drugs(xlsx_path="42_medications.xlsx"):
    """Extract the 42 medication names from the Japanese-4D species-level sheet."""
    japanese_sheet_candidates = ["S9", "Our genus_medication_multi"]
    jap_raw = None
    used_sheet = None

    for sheet_name in japanese_sheet_candidates:
        try:
            jap_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
            used_sheet = sheet_name
            break
        except ValueError:
            continue

    if jap_raw is None:
        raise ValueError(
            "Could not find a usable Japanese-4D worksheet in 42_medications.xlsx. "
            "Tried: " + ", ".join(japanese_sheet_candidates)
        )

    drugs = (
        pd.Series(jap_raw.iloc[0, 1:85:2])
        .fillna("")
        .astype(str)
        .str.strip()
    )
    drugs = [d for d in drugs if d and d.lower() != "nan"]

    if len(drugs) != 42:
        raise ValueError(f"Expected 42 medications from sheet '{used_sheet}', found {len(drugs)}")

    print(f"Loaded 42 target medications from {xlsx_path} (sheet: {used_sheet})")
    return drugs, jap_raw, used_sheet

print("==================================================")
print("PART 1: PARALLEL MULTIVARIATE REGRESSION (MetaCardis)")
print("==================================================")
start_time = time.time()

# Load and freeze the exact 42-drug panel used for Figure 4 across both cohorts.
target_42_drugs, jap_raw, used_japanese_sheet = load_target_42_drugs("42_medications.xlsx")

# --- 1. Load MetaCardis Data ---
all_abundance_df = load_all_abundance_df()
drug_matrix = pd.read_csv("drug_matrix.csv").fillna(0)

# Make column names unique
cols = pd.Series(all_abundance_df.columns)
for dup in cols[cols.duplicated()].unique(): 
    cols[cols[cols == dup].index.values.tolist()] = [dup + '.' + str(i) if i != 0 else dup for i in range(sum(cols == dup))]
all_abundance_df.columns = cols

all_species = all_abundance_df.columns.tolist()
all_abundance_df['Sample'] = all_abundance_df.index
available_meta_drugs = [col for col in drug_matrix.columns if col != 'Sample']
selected_drugs = [d for d in target_42_drugs if d in available_meta_drugs]

if len(selected_drugs) != 42:
    missing_from_meta = sorted(set(target_42_drugs) - set(selected_drugs))
    raise ValueError(
        "Figure 4 requires all 42 target medications in drug_matrix.csv. Missing: "
        + ", ".join(missing_from_meta)
    )

print(f"Using {len(selected_drugs)} harmonized medications for MetaCardis regression.")

analysis_df_all = pd.merge(all_abundance_df, drug_matrix[['Sample'] + selected_drugs], on='Sample', how='inner')

# --- 2. Parallel Regression ---
print(f"Running highly parallel OLS models for {len(all_species)} species...")
X = analysis_df_all[selected_drugs]
X = sm.add_constant(X) 
X_np = X.values 
drug_names = X.columns.tolist()

def fit_ols_fast(sp_name, y_values):
    model = sm.OLS(y_values, X_np).fit()
    return sp_name, model.params, model.pvalues

parallel_results = Parallel(n_jobs=-1)(
    delayed(fit_ols_fast)(sp, analysis_df_all[sp].values) for sp in all_species
)

results_list = []
for sp_name, params, pvals in parallel_results:
    for idx, drug in enumerate(drug_names):
        if drug == 'const': continue
        results_list.append({
            'Species': sp_name,
            'Drug': drug,
            'Estimate': params[idx],
            'Pvalue': pvals[idx],
            'Cohort': 'MetaCardis'
        })

all_results_mc = pd.DataFrame(results_list).dropna(subset=['Pvalue'])
print("-> MetaCardis models complete and NA values removed!")

print("==================================================")
print("PART 2: PARSING JAPANESE-4D EXCEL DATA")
print("==================================================")

# --- 3. Extract Japanese-4D from Excel ---
print(f"Using Japanese-4D sheet: {used_japanese_sheet} (shape={jap_raw.shape})")

# Row index 0 has the drug names. Column 0 has the species.
drug_names_j4d = jap_raw.iloc[0, 1:85:2].fillna("").astype(str).values
jap_species = (
    jap_raw.iloc[3:, 0]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .values
)

# Remove empty/non-data species labels while preserving row alignment.
valid_species_mask = np.array([
    (sp != "") and (sp != "nan") and (not sp.startswith("Name_as_per"))
    for sp in jap_species
])

j4d_list = []
for i, drug in enumerate(drug_names_j4d):
    if drug in selected_drugs:
        est_col = 1 + 2*i
        pval_col = 2 + 2*i
        est_vals = pd.to_numeric(jap_raw.iloc[3:, est_col], errors='coerce').values
        pval_vals = pd.to_numeric(jap_raw.iloc[3:, pval_col], errors='coerce').values

        species_filtered = jap_species[valid_species_mask]
        est_filtered = est_vals[valid_species_mask]
        pval_filtered = pval_vals[valid_species_mask]
        
        df_j4d = pd.DataFrame({
            'Species': species_filtered,
            'Drug': drug,
            'Estimate': est_filtered,
            'Pvalue': pval_filtered,
            'Cohort': 'Japanese-4D'
        })
        j4d_list.append(df_j4d)

all_results_j4d = pd.concat(j4d_list, ignore_index=True).dropna(subset=['Pvalue'])
print(f"-> Japanese-4D parsed successfully! Found {len(all_results_j4d)} valid NA-free models.")

# --- 4. Merge and Export ---
print("==================================================")
print("PART 3: EXPORTING CLEANED RESULTS")
print("==================================================")

# Combine both cohorts into a master file (used by the R script)
all_results_combined = pd.concat([all_results_mc, all_results_j4d], ignore_index=True)
all_results_combined.to_csv("All_Cohorts_Regression_Results.csv", index=False)

# Filter out the PI's target species (Streptococcus, Veillonella, Rothia)
target_species_all = [sp for sp in all_results_combined['Species'].unique() if sp.startswith(('Streptococcus_', 'Veillonella_', 'Rothia_'))]
pi_stats = all_results_combined[all_results_combined['Species'].isin(target_species_all)].sort_values(by=['Cohort', 'Pvalue'])

# Export the CSV for the Professor
pi_stats.to_csv("Raw_Multivariate_Regression_Results.csv", index=False)

print("-> Saved 'All_Cohorts_Regression_Results.csv' (For R Script Heatmap)")
print("-> Saved 'Raw_Multivariate_Regression_Results.csv' (For Professor Ghosh)")
print(f"Python processing complete in {round(time.time() - start_time, 2)} seconds!")

In [1]:
# ==============================================================================
# SCRIPT 1 (PYTHON): Parallel Regression & Japanese-4D Integration
# ==============================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from joblib import Parallel, delayed
import time
import os


def load_all_abundance_df():
    """Load AllCombinedSpProfile from CSV, or fall back to RData when CSV is empty."""
    csv_path = "AllCombinedSpProfile.csv"

    if os.path.exists(csv_path):
        try:
            csv_df = pd.read_csv(csv_path, index_col=0)
            if not csv_df.empty and csv_df.shape[1] > 0:
                print(f"Loaded abundance matrix from {csv_path}")
                return csv_df.fillna(0)
            print(f"{csv_path} exists but is empty; falling back to RData...")
        except pd.errors.EmptyDataError:
            print(f"{csv_path} is empty; falling back to RData...")
    else:
        print(f"{csv_path} not found; falling back to RData...")

    try:
        import pyreadr
    except ImportError as exc:
        raise ImportError(
            "pyreadr is required to read AllCombinedSpProfile from RData. "
            "Install with: pip install pyreadr"
        ) from exc

    rdata_candidates = [
        "Required_Data_FEBS_Pipeline.RData",
        "Required_Data_FEBS_Pipepline.RData"
    ]

    for rdata_path in rdata_candidates:
        if not os.path.exists(rdata_path):
            continue

        print(f"Trying to load abundance matrix from {rdata_path}...")
        r_objects = pyreadr.read_r(rdata_path)

        if "AllCombinedSpProfile" not in r_objects:
            continue

        all_abundance_df = r_objects["AllCombinedSpProfile"]
        if not isinstance(all_abundance_df, pd.DataFrame) or all_abundance_df.empty:
            raise ValueError(
                f"AllCombinedSpProfile in {rdata_path} is missing or empty."
            )

        print(f"Loaded abundance matrix from {rdata_path} (object: AllCombinedSpProfile)")
        return all_abundance_df.fillna(0)

    raise FileNotFoundError(
        "Could not load AllCombinedSpProfile. Checked AllCombinedSpProfile.csv and "
        "RData files: Required_Data_FEBS_Pipeline.RData / Required_Data_FEBS_Pipepline.RData"
    )


def load_target_42_drugs(xlsx_path="42_medications.xlsx"):
    """Extract the 42 medication names from the Japanese-4D species-level sheet."""
    japanese_sheet_candidates = ["S9", "Our genus_medication_multi"]
    jap_raw = None
    used_sheet = None

    for sheet_name in japanese_sheet_candidates:
        try:
            jap_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
            used_sheet = sheet_name
            break
        except ValueError:
            continue

    if jap_raw is None:
        raise ValueError(
            "Could not find a usable Japanese-4D worksheet in 42_medications.xlsx. "
            "Tried: " + ", ".join(japanese_sheet_candidates)
        )

    drugs = (
        pd.Series(jap_raw.iloc[0, 1:85:2])
        .fillna("")
        .astype(str)
        .str.strip()
    )
    drugs = [d for d in drugs if d and d.lower() != "nan"]

    if len(drugs) != 42:
        raise ValueError(f"Expected 42 medications from sheet '{used_sheet}', found {len(drugs)}")

    print(f"Loaded 42 target medications from {xlsx_path} (sheet: {used_sheet})")
    return drugs, jap_raw, used_sheet

In [2]:
print("==================================================")
print("PART 1: PARALLEL MULTIVARIATE REGRESSION (MetaCardis)")
print("==================================================")
start_time = time.time()

# Load and freeze the exact 42-drug panel used for Figure 4 across both cohorts.
target_42_drugs, jap_raw, used_japanese_sheet = load_target_42_drugs("42_medications.xlsx")

# --- 1. Load MetaCardis Data ---
all_abundance_df = load_all_abundance_df()
drug_matrix = pd.read_csv("drug_matrix.csv").fillna(0)

# Make column names unique
cols = pd.Series(all_abundance_df.columns)
for dup in cols[cols.duplicated()].unique(): 
    cols[cols[cols == dup].index.values.tolist()] = [dup + '.' + str(i) if i != 0 else dup for i in range(sum(cols == dup))]
all_abundance_df.columns = cols

all_species = all_abundance_df.columns.tolist()
all_abundance_df['Sample'] = all_abundance_df.index
available_meta_drugs = [col for col in drug_matrix.columns if col != 'Sample']
selected_drugs = [d for d in target_42_drugs if d in available_meta_drugs]

if len(selected_drugs) != 42:
    missing_from_meta = sorted(set(target_42_drugs) - set(selected_drugs))
    raise ValueError(
        "Figure 4 requires all 42 target medications in drug_matrix.csv. Missing: "
        + ", ".join(missing_from_meta)
    )

print(f"Using {len(selected_drugs)} harmonized medications for MetaCardis regression.")

analysis_df_all = pd.merge(all_abundance_df, drug_matrix[['Sample'] + selected_drugs], on='Sample', how='inner')

# --- 2. Parallel Regression ---
print(f"Running highly parallel OLS models for {len(all_species)} species...")
X = analysis_df_all[selected_drugs]
X = sm.add_constant(X) 
X_np = X.values 
drug_names = X.columns.tolist()

def fit_ols_fast(sp_name, y_values):
    model = sm.OLS(y_values, X_np).fit()
    return sp_name, model.params, model.pvalues

parallel_results = Parallel(n_jobs=-1)(
    delayed(fit_ols_fast)(sp, analysis_df_all[sp].values) for sp in all_species
)

results_list = []
for sp_name, params, pvals in parallel_results:
    for idx, drug in enumerate(drug_names):
        if drug == 'const': continue
        results_list.append({
            'Species': sp_name,
            'Drug': drug,
            'Estimate': params[idx],
            'Pvalue': pvals[idx],
            'Cohort': 'MetaCardis'
        })

all_results_mc = pd.DataFrame(results_list).dropna(subset=['Pvalue'])
print("-> MetaCardis models complete and NA values removed!")

PART 1: PARALLEL MULTIVARIATE REGRESSION (MetaCardis)
Loaded 42 target medications from 42_medications.xlsx (sheet: S9)
Loaded abundance matrix from AllCombinedSpProfile.csv
Using 42 harmonized medications for MetaCardis regression.
Running highly parallel OLS models for 7436 species...
-> MetaCardis models complete and NA values removed!


In [5]:
print(all_abundance_df)
print(all_species)


                            Parabacteroides_goldsteinii  \
100143                                         0.000000   
7899                                           0.000000   
DA01441                                        0.000000   
ERAS6_Dag4opt                                  0.000000   
ERR1513668                                     0.000000   
...                                                 ...   
ER0369                                         0.283854   
parkinsonfecalbacteria.P11                     0.352859   
S000060329                                     0.360053   
MHH249                                         0.398137   
G440606491                                     0.472321   

                            Bacteroides_thetaiotaomicron  \
100143                                          0.000000   
7899                                            0.000000   
DA01441                                         0.000000   
ERAS6_Dag4opt                                   0.0

In [6]:
print(drug_matrix)

      Unnamed: 0        Sample  Omeprazole  Pantoprazole  Lansoprazole  \
0              4  M0x10MCx1548           0             0             0   
1              5  M0x10MCx1272           0             0             0   
2              7  M0x10MCx3346           0             0             0   
3              8  M0x10MCx1844           0             0             0   
4              9  M0x10MCx2824           0             1             0   
...          ...           ...         ...           ...           ...   
1826        2169  M0x30MCx3091           0             0             0   
1827        2170  M0x30MCx1996           0             0             0   
1828        2171  M0x30MCx1308           0             0             0   
1829        2172  M0x30MCx3340           0             0             0   
1830        2173  M0x30MCx2168           0             0             0   

      Rabeprazole  Esomeprazole  Metformin  Glibenclamide  Tolbutamide  ...  \
0               0             0 

In [3]:
print("==================================================")
print("PART 2: PARSING JAPANESE-4D EXCEL DATA")
print("==================================================")

# --- 3. Extract Japanese-4D from Excel ---
print(f"Using Japanese-4D sheet: {used_japanese_sheet} (shape={jap_raw.shape})")

# Row index 0 has the drug names. Column 0 has the species.
drug_names_j4d = jap_raw.iloc[0, 1:85:2].fillna("").astype(str).values
jap_species = (
    jap_raw.iloc[3:, 0]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .values
)

# Remove empty/non-data species labels while preserving row alignment.
valid_species_mask = np.array([
    (sp != "") and (sp != "nan") and (not sp.startswith("Name_as_per"))
    for sp in jap_species
])

j4d_list = []
for i, drug in enumerate(drug_names_j4d):
    if drug in selected_drugs:
        est_col = 1 + 2*i
        pval_col = 2 + 2*i
        est_vals = pd.to_numeric(jap_raw.iloc[3:, est_col], errors='coerce').values
        pval_vals = pd.to_numeric(jap_raw.iloc[3:, pval_col], errors='coerce').values

        species_filtered = jap_species[valid_species_mask]
        est_filtered = est_vals[valid_species_mask]
        pval_filtered = pval_vals[valid_species_mask]
        
        df_j4d = pd.DataFrame({
            'Species': species_filtered,
            'Drug': drug,
            'Estimate': est_filtered,
            'Pvalue': pval_filtered,
            'Cohort': 'Japanese-4D'
        })
        j4d_list.append(df_j4d)

all_results_j4d = pd.concat(j4d_list, ignore_index=True).dropna(subset=['Pvalue'])
print(f"-> Japanese-4D parsed successfully! Found {len(all_results_j4d)} valid NA-free models.")


PART 2: PARSING JAPANESE-4D EXCEL DATA
Using Japanese-4D sheet: S9 (shape=(163, 85))
-> Japanese-4D parsed successfully! Found 6678 valid NA-free models.


In [7]:
reg_result = pd.read_csv("Raw_Multivariate_Regression_Results.csv")

In [8]:
print(reg_result)

                                    Species          Drug      Estimate  \
0        Streptococcus_salivarius_[ID:0199]  Lansoprazole  6.722787e-01   
1     Streptococcus_parasanguinis_[ID:0144]  Lansoprazole  4.941007e-01   
2        Streptococcus_salivarius_[ID:0199]  Esomeprazole  6.464518e-01   
3     Streptococcus_parasanguinis_[ID:0144]  Esomeprazole  4.087873e-01   
4      Streptococcus_vestibularis_[ID:0198]  Lansoprazole  3.952781e-01   
...                                     ...           ...           ...   
2263          Streptococcus_sp_NLAE_zl_C503   Liraglutide  1.968492e-10   
2264              Streptococcus_macedonicus     Diltiazem -1.658134e-09   
2265             Streptococcus_massiliensis   Repaglinide -1.411546e-09   
2266             Streptococcus_massiliensis    Alogliptin  3.873963e-09   
2267                Veillonella_sp_T11011_6    Alogliptin -2.570878e-08   

            Pvalue       Cohort  
0     2.225848e-84  Japanese-4D  
1     1.063515e-65  Japanese-4D

In [11]:
val = reg_result.iloc[:, 0].unique()

In [12]:
len(val)

54

In [16]:
uni_species = pd.read_excel("Focal_Species_Cohort_Medication_Table_TSG.xlsx")

In [21]:
val1 = uni_species.iloc[:, 2].unique()

In [22]:
len(val1)

28